# N07 BPE 分词器

> **对应理论篇**：[`04-分词器与词表.md`](../../01-基础/04-分词器与词表.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/tokenizer.py`（GPT-2 BPE + Lean/数学专用 token）、`tokenizer.py:14` `SPECIAL_TOKENS`
> **Colab**：CPU 可跑，约 40 秒。

## 学习目标
1. 理解字符级分词与 BPE 的区别、词表大小与序列长度的权衡；
2. 用 `tokenizers` 训练一个 byte-level BPE，统计压缩率；
3. 理解为什么 Lean/数学语料要加专用 token（如 `<|tactic|>`、`<|bin_XX|>`）。


In [1]:
!pip -q install torch tokenizers 2>/dev/null; echo ok

ok


In [2]:
import os, sys
def find_code_dir():
    for c in ["../../01-基础/code/from_scratch", "01-基础/code/from_scratch",
              "../01-基础/code/from_scratch", "code/from_scratch"]:
        if os.path.exists(os.path.join(c, "configs.py")):
            return os.path.abspath(c)
    cur = os.getcwd()
    for _ in range(8):
        p = os.path.join(cur, "01-基础", "code", "from_scratch")
        if os.path.exists(os.path.join(p, "configs.py")):
            return p
        cur = os.path.dirname(cur)
    raise FileNotFoundError("找不到 from_scratch 代码目录")
sys.path.insert(0, find_code_dir())


## 理论回顾：为什么要子词分词？

字符级分词词表小（~几十到几百）但序列长；词级分词序列短但词表爆炸、有 OOV。
**BPE（Byte-Pair Encoding）** 折中：从字节出发，反复合并出现频率最高的相邻符号对。

$$
(a,b)^\star=\arg\max_{(a,b)} \mathrm{count}(a\,b),\qquad \text{vocab} \leftarrow \text{vocab}\cup\{ab\}
$$

**压缩率** $\rho=\frac{\#\text{chars}}{\#\text{tokens}}$ 越大越好（同样上下文容纳更多字符）。
nanoproof 在 GPT-2 BPE 上**追加**了一批数学/Lean 专用 token（`tokenizer.py:14`），
让 `∀ ∃ ≤ Γ <|tactic|> <|bin_01|>…` 等各占一个 token，既缩短序列又让价值分箱
`<|bin_XX|>` 有确定含义。


## 实验 1：字符级 vs BPE 压缩率

In [3]:
from data import load_text, CharTokenizer

text = load_text(download=False)
print("字符数:", len(text))

char_tok = CharTokenizer.from_text(text)
char_ids = char_tok.encode(text)
print(f"字符级: vocab={char_tok.vocab_size}, tokens={len(char_ids)}, "
      f"压缩率={len(text)/len(char_ids):.2f}")

try:
    from tokenizers import ByteLevelBPETokenizer
    bpe = ByteLevelBPETokenizer()
    bpe.train_from_iterator([text], vocab_size=512,
                            special_tokens=["<pad>", "<eos>", "<bos>", "<unk>"])
    bpe_ids = bpe.encode(text).ids
    print(f"BPE   : vocab={bpe.get_vocab_size()}, tokens={len(bpe_ids)}, "
          f"压缩率={len(text)/len(bpe_ids):.2f}")
except ImportError as e:
    print("未安装 tokenizers，跳过 BPE 统计:", e)


字符数: 1209
字符级: vocab=56, tokens=1209, 压缩率=1.00


BPE   : vocab=362, tokens=489, 压缩率=2.47


## 实验 2：BPE 的 encode / decode 往返

In [4]:
try:
    samples = [
        "theorem add_comm (a b : Nat) : a + b = b + a := by",
        "induction a with | zero => simp | succ a ih => rw [ih]",
        "∀ x : ℕ, x + 0 = x",
    ]
    for s in samples:
        ids = bpe.encode(s).ids
        print("原文:", s)
        print("ids :", ids)
        print("子词:", bpe.encode(s).tokens)
        print("解码:", bpe.decode(ids))
        print("-" * 50)
except NameError:
    print("BPE 未训练（缺 tokenizers），跳过。")


原文: theorem add_comm (a b : Nat) : a + b = b + a := by
ids : [289, 331, 66, 360, 265, 68, 261, 260, 285, 12, 260, 264, 290, 261, 262, 261, 290, 264, 279, 280]
子词: ['theorem', 'Ġadd', '_', 'comm', 'Ġ(', 'a', 'Ġb', 'Ġ:', 'ĠNat', ')', 'Ġ:', 'Ġa', 'Ġ+', 'Ġb', 'Ġ=', 'Ġb', 'Ġ+', 'Ġa', 'Ġ:=', 'Ġby']
解码: theorem add_comm (a b : Nat) : a + b = b + a := by
--------------------------------------------------
原文: induction a with | zero => simp | succ a ih => rw [ih]
ids : [76, 81, 306, 303, 264, 312, 275, 317, 276, 277, 275, 323, 264, 286, 276, 356, 300, 340, 64]
子词: ['i', 'n', 'duc', 'tion', 'Ġa', 'Ġwith', 'Ġ|', 'Ġzero', 'Ġ=>', 'Ġsimp', 'Ġ|', 'Ġsucc', 'Ġa', 'Ġih', 'Ġ=>', 'Ġrw', 'Ġ[', 'ih', ']']
解码: induction a with | zero => simp | succ a ih => rw [ih]
--------------------------------------------------
原文: ∀ x : ℕ, x + 0 = x
ids : [162, 234, 226, 348, 260, 301, 230, 247, 15, 348, 290, 346, 262, 348]
子词: ['â', 'Ī', 'Ģ', 'Ġx', 'Ġ:', 'Ġâ', 'Ħ', 'ķ', ',', 'Ġx', 'Ġ+', 'Ġ0', 'Ġ=', 'Ġx']
解码: ∀ x : ℕ, x 

## 实验 3：专用 token 与价值分箱

In [5]:
# 演示：为什么数学符号需要专用 token
symbols = ["∀", "∃", "≤", "ℕ", "ℝ", "α", "<|tactic|>", "<|bin_01|>"]
try:
    for s in symbols:
        ids = bpe.encode(s).ids
        print(f"{s!r:14s} -> {len(ids)} 个 BPE token: {ids}")
    print("\n如果符号被拆成多个 token，模型要为'同一个数学对象'学多次；")
    print("nanoproof 因此把它们加入 SPECIAL_TOKENS（tokenizer.py:14），强制单 token。")
except NameError:
    # 无 tokenizers 时用字符级演示
    for s in symbols:
        print(f"{s!r:14s} -> 字符级 {len(char_tok.encode(s))} 个 token")
    print("nanoproof 把数学符号加入 SPECIAL_TOKENS（tokenizer.py:14），强制单 token。")


'∀'            -> 3 个 BPE token: [162, 234, 226]
'∃'            -> 3 个 BPE token: [162, 234, 229]
'≤'            -> 3 个 BPE token: [162, 235, 101]
'ℕ'            -> 3 个 BPE token: [162, 230, 247]
'ℝ'            -> 3 个 BPE token: [162, 230, 255]
'α'            -> 2 个 BPE token: [142, 113]
'<|tactic|>'   -> 9 个 BPE token: [31, 95, 87, 68, 70, 293, 70, 95, 33]
'<|bin_01|>'   -> 10 个 BPE token: [31, 95, 69, 76, 81, 66, 19, 20, 95, 33]

如果符号被拆成多个 token，模型要为'同一个数学对象'学多次；
nanoproof 因此把它们加入 SPECIAL_TOKENS（tokenizer.py:14），强制单 token。


## 思考题

1. 词表变大有哪些代价？（embedding/lm_head 参数量、softmax 计算、长尾 token 欠训练）
2. BPE 的合并次数与最终词表大小是什么关系？
3. nanoproof 的 `<|bin_01|>…<|bin_64|>` 为什么必须在 tokenizer 阶段就固定？
   （提示：`common.py:70` `num_value_bins=64` 与 `tokenizer.py:19` 必须一致）

## 改动实验（任选 2–3 个）
1. 把 `vocab_size` 从 512 改成 256 / 1024，观察压缩率变化。
2. 给 BPE 加一批 Lean 专用 token，再测压缩率。
3. 对手写 BPE 合并 k 步后，直接比较 token 序列长度。
4. 用同一段文本对比「字符级 + RoPE」与「BPE」训练 loss 曲线（改 `prepare` 的 tokenizer）。

## 预期输出
```
字符数: ...
字符级: vocab=..., tokens=..., 压缩率=1.00
BPE   : vocab=512, tokens=..., 压缩率=2~3
'theorem ...' 的 ids/子词/解码（往返一致）
```
压缩率通常随语料增大而提高；小语料上 BPE 优势有限。
